In [1]:
from datasets import load_dataset
ds = load_dataset("gamma-lab-umd/MMAU-Pro")
ds = ds['test']

/home/u1501463/miniconda3/envs/gemini/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import json
import numpy as np
from tqdm import tqdm
import soundfile as sf
output_path = '../MMAU_pro_subset.json'
data_root = '/work/u1501463/MMAU-Pro/'
subset_samples = []

def process_audio_path(item):
    processed_audio_root = './MMAU_human_voice_amplify/tool_outputs'
    audio_name = 'tool_result_denoise_.wav'
    item_id = item['id']
    denoised_audio_path = os.path.join(processed_audio_root, item_id, audio_name)
    return denoised_audio_path

def get_audio_duration(path: str):
    with sf.SoundFile(path) as f:
        frames = f.frames
        sr = f.samplerate
    return frames / sr

time_limit = 30.0

for sample in tqdm(ds):
    if len(sample['audio_path']) != 1:
        continue
    # sample['question'] = sample.pop('question')
    key_pairs = [
        ('question', 'question'),
        ('choices', 'choice'),
        ('answer', 'answer'),
        ('id', 'id'),
        ('category', 'category'),
        ('sub-cat', 'question_type'),
        ('perceptual_skills', 'perceptual_skills'),
        ('reasoning_skills', 'reasoning_skills'),
    ]

    audio_path = sample['audio_path'][0]
    audio_path = os.path.join(data_root, audio_path)

    audio_duration = get_audio_duration(audio_path)
    if audio_duration > time_limit:
        continue

    subset_sample = {new_key: sample[old_key] for old_key, new_key in key_pairs}
    subset_sample['audio_path'] = audio_path
    subset_samples.append(subset_sample)

with open(output_path, 'w') as f:
    json.dump(subset_samples, f, indent=4)

len(subset_samples)

100%|██████████| 5305/5305 [00:02<00:00, 2150.14it/s]


2013